# 01 · Tablas: celdas, infraestructura (OpenStreetMap), accidentes y clima

**Qué hace**, en 6 pasos:
1. Lee (o descarga, la primera vez) las calles de **OpenStreetMap**.
2. Parte las avenidas principales en **celdas geohash** de ~150 × 150 m.
3. Calcula **19 variables de infraestructura** por celda (carriles, intersecciones, puentes, paradas…).
4. Asigna cada accidente a su celda.
5. Pone a cada celda su municipio y marca las de **buen registro**.
6. Descarga la **lluvia y temperatura** por hora (Open-Meteo) y las resume por día.

**Entra:** `data/accidentes_zmg.csv` (del 00)
**Sale:** `data/celdas.csv`, `data/accidentes_celdas.csv`, `data/clima_horario.csv`, `data/clima_diario.csv`

⏱️ Si ya existe `data/osm/zmg_osm.gpkg`, tarda unos minutos. Si no, la descarga de OSM puede tardar horas (se puede cortar y reanudar).

In [2]:
import re, time, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, shapely, pygeohash as pgh, requests
from shapely.geometry import box

# ----- Parámetros (todo lo que se puede ajustar está aquí) -----
UTM = "EPSG:32613"                    # coordenadas en METROS (UTM zona 13 norte), para medir distancias
GH_PREC = 7                           # geohash de 7 caracteres ≈ 143 × 152 m
BUFFER_VIA_M = 40                     # un accidente "es de" una avenida si está a ≤ 40 m
RADIO_ENTORNO_M = 100                 # paradas, comercios, etc. se cuentan a 100 m de la celda
CENTRO_GDL = (20.6767, -103.3475)     # catedral, para la distancia al centro
CLASES_AVENIDA = ["motorway", "trunk", "primary", "secondary"]          # avenidas principales en OSM
ORDEN_CLASE = {"secondary": 1, "primary": 2, "trunk": 3, "motorway": 4}
PATRON_LM = r"L[OÓ]PEZ MATEOS|JIQUILPAN"   # al sur del Periférico, OSM la llama Carretera Jiquilpan
MUNICIPIOS_BUEN_REGISTRO = [39, 120, 98]   # Guadalajara, Zapopan, Tlaquepaque (ver paso 5)
GPKG = Path("data/osm/zmg_osm.gpkg")

acc = pd.read_csv("data/accidentes_zmg.csv", parse_dates=["FECHA"])
print(f"Accidentes ZMG: {len(acc):,}")

Accidentes ZMG: 24,872


## Paso 1. Calles de OpenStreetMap
Se usan 4 capas:

| Capa | Qué es | Para qué |
|---|---|---|
| `vias` | tramos de avenidas principales, con carriles, velocidad, puente, túnel | variables de la avenida |
| `aristas` | todas las calles (red para autos) | calles que cruzan, glorietas |
| `nodos` | esquinas de la red, con cuántas calles llegan | intersecciones, semáforos |
| `pois` | paradas, cruces peatonales, semáforos, comercios, servicios | entorno a 100 m |

**No necesitas leer el código de la descarga:** solo corre si el archivo no existe. Pide la información por pedazos a los servidores públicos de OSM y, si uno falla, prueba con otro.

In [3]:
if GPKG.exists():
    print("Ya existe", GPKG, "→ se lee (no se descarga nada)")
else:
    import osmnx as ox
    from osmnx import _http, _overpass
    GPKG.parent.mkdir(parents=True, exist_ok=True)
    ox.settings.use_cache, ox.settings.cache_folder = True, str(GPKG.parent / "cache")
    ox.settings.max_query_area_size = 25 * 1000 * 1000      # pedazos de ≤ 25 km²
    ox.settings.requests_timeout, ox.settings.overpass_rate_limit = 300, False
    SERVIDORES = ["https://overpass-api.de/api", "https://overpass.private.coffee/api",
                  "https://maps.mail.ru/osm/tools/overpass/api", "https://overpass.kumi.systems/api"]
    # caché que no depende del servidor + cada pedazo prueba los servidores hasta que uno responda
    _resolver = _http._resolve_cache_filepath
    def sin_servidor(k):
        for u in SERVIDORES:
            k = k.replace(u, SERVIDORES[0])
        return k
    _http._resolve_cache_filepath = lambda key, extension="json": _resolver(sin_servidor(key), extension)
    _http._check_cache = lambda key: next((r for r in [_resolver(sin_servidor(key)), _resolver(key)] if r.is_file()), None)
    _pedir = _overpass._overpass_request
    def pedir_con_respaldo(data):
        for vuelta in range(3):
            for url in SERVIDORES:
                ox.settings.overpass_url = url
                try:
                    return _pedir(data)
                except (requests.RequestException, ValueError):
                    print("   ✗", url.split("/")[2], "falló; se pide a otro servidor…")
            time.sleep(120)
        raise RuntimeError("Overpass no responde; vuelve a correr más tarde (lo bajado queda en caché).")
    _overpass._overpass_request = pedir_con_respaldo

    texto = lambda v: ";".join(map(str, v)) if isinstance(v, (list, tuple)) else ("" if pd.isna(v) else str(v))
    a = acc[acc.LATITUD.between(*acc.LATITUD.quantile([.005, .995])) & acc.LONGITUD.between(*acc.LONGITUD.quantile([.005, .995]))]
    pts = gpd.GeoSeries(gpd.points_from_xy(a.LONGITUD, a.LATITUD), crs="EPSG:4326").to_crs(UTM)
    zona = gpd.GeoSeries([pts.union_all().convex_hull.buffer(500)], crs=UTM).to_crs("EPSG:4326").iloc[0]

    print("[1/3] Avenidas principales…")
    v = ox.features_from_polygon(zona, tags={"highway": CLASES_AVENIDA + [c + "_link" for c in CLASES_AVENIDA]})
    v = v[v.geom_type.isin(["LineString", "MultiLineString"])].reset_index()
    cols = ["name", "highway", "lanes", "maxspeed", "bridge", "tunnel", "oneway", "junction", "ref"]
    for c in cols:
        v[c] = v[c].map(texto) if c in v else ""
    v[["id"] + cols + ["geometry"]].to_file(GPKG, layer="vias")
    print("[2/3] Red vial completa…")
    G = ox.graph_from_polygon(zona, network_type="drive", simplify=True, retain_all=True)
    n, e = ox.graph_to_gdfs(G)
    n = n.reset_index().reindex(columns=["osmid", "street_count", "highway", "geometry"]); n["highway"] = n.highway.map(texto)
    e = e.reset_index()
    for c in ["name", "highway", "lanes", "junction"]:
        e[c] = e[c].map(texto) if c in e else ""
    n.to_file(GPKG, layer="nodos"); e[["u", "v", "name", "highway", "lanes", "junction", "geometry"]].to_file(GPKG, layer="aristas")
    print("[3/3] Paradas, cruces peatonales, semáforos y comercios…")
    p = ox.features_from_polygon(zona, tags={"highway": ["bus_stop", "crossing", "traffic_signals"],
                                             "public_transport": ["platform", "stop_position"], "shop": True, "amenity": True}).reset_index()
    col = lambda c: p[c] if c in p else pd.Series(np.nan, index=p.index)
    p["tipo"] = np.select([col("highway").eq("bus_stop") | col("public_transport").isin(["platform", "stop_position"]),
                           col("highway").eq("crossing"), col("highway").eq("traffic_signals"), col("shop").notna(), col("amenity").notna()],
                          ["parada", "cruce_peatonal", "semaforo", "comercio", "servicio"], default="otro")
    gpd.GeoDataFrame(p[["tipo"]], geometry=p.geometry.representative_point(), crs=p.crs).to_file(GPKG, layer="pois")

vias, nodos, aristas, pois = [gpd.read_file(GPKG, layer=c).to_crs(UTM) for c in ["vias", "nodos", "aristas", "pois"]]
print(f"vias: {len(vias):,} · aristas: {len(aristas):,} · nodos: {len(nodos):,} · pois: {len(pois):,}")

Ya existe data/osm/zmg_osm.gpkg → se lee (no se descarga nada)
vias: 6,434 · aristas: 238,753 · nodos: 99,852 · pois: 34,152


**Tres ayudantes** para leer las etiquetas de OSM:
- `a_numero("2;3")` → 3: carriles y velocidad a veces vienen como texto.
- `nombre_canonico(...)`: "Av. Adolfo López Mateos Sur" y "Carretera Jiquilpan" → "lopez mateos", para que una avenida con varios nombres cuente como una sola.

In [4]:
def a_numero(v):
    nums = re.findall(r"\d+(?:\.\d+)?", str(v) if pd.notna(v) else "")
    return max(map(float, nums)) if nums else np.nan

def nombre_canonico(n):
    n = str(n).split(";")[0].strip() if pd.notna(n) else ""
    if not n:
        return ""
    if re.search(PATRON_LM, n, re.I):
        return "lopez mateos"
    n = unicodedata.normalize("NFKD", n.lower()).encode("ascii", "ignore").decode()
    n = re.sub(r"^(lateral( de(l| la)?)?\s+)", "", n)
    n = re.sub(r"^(avenida|av\.?|calzada|calz\.?|prolongacion|boulevard|blvd\.?|anillo|carretera|calle)\s+", "", n)
    n = re.sub(r"(?<!del )(?<!de )\b(norte|sur|oriente|poniente|nte|pte|ote)\b\.?", " ", n)   # "División del Norte" se conserva
    return re.sub(r"\s+", " ", n).strip()

print(nombre_canonico("Avenida Adolfo López Mateos Sur"), "|", nombre_canonico("Calzada Lázaro Cárdenas"),
      "|", nombre_canonico("Avenida División del Norte"))

lopez mateos | lazaro cardenas | division del norte


## Paso 2. Celdas geohash sobre las avenidas principales
Se recorre cada avenida principal poniendo un punto cada 20 m. A cada punto se le saca su código geohash de 7 caracteres, y **cada código distinto es una celda**.

In [5]:
avenidas = vias[~vias.highway.str.endswith("_link")]                      # sin rampas
puntos = np.concatenate([shapely.line_interpolate_point(g, np.arange(0, g.length + 20, 20)) for g in avenidas.geometry])
puntos = gpd.GeoSeries(puntos, crs=UTM).to_crs("EPSG:4326")
codigos = sorted({pgh.encode(y, x, GH_PREC) for x, y in zip(puntos.x, puntos.y)})

def caja_geohash(c):
    lat, lon, dlat, dlon = pgh.decode_exactly(c)
    return c, lat, lon, box(lon - dlon, lat - dlat, lon + dlon, lat + dlat)

cel = gpd.GeoDataFrame([caja_geohash(c) for c in codigos], columns=["gh7", "lat", "lon", "geometry"], crs="EPSG:4326").to_crs(UTM)
print(f"Celdas sobre avenidas principales: {len(cel):,}")
print("Ejemplo:", cel.gh7.iloc[0], "→ sus primeros 5 caracteres", cel.gh7.iloc[0][:5], "son el bloque de ~5 km que la contiene")

Celdas sobre avenidas principales: 5,775
Ejemplo: 9ewkmvr → sus primeros 5 caracteres 9ewkm son el bloque de ~5 km que la contiene


## Paso 3. Variables de infraestructura por celda
### 3a. La avenida de la celda
La **avenida principal** de una celda es la que tiene más metros dentro de ella. De esa avenida salen:

| Variable | Qué es |
|---|---|
| `clase_via` | 1 secundaria, 2 primaria, 3 troncal, 4 autopista |
| `carriles_via` | carriles (máximo de sus tramos) |
| `vel_max_via` | velocidad máxima |
| `puente`, `tunel` | 1 si la avenida pasa por puente o paso deprimido en la celda |
| `n_avenidas`, `cruce_avenidas` | cuántas avenidas principales distintas hay; 1 si son 2 o más |
| `lateral`, `rampas` | 1 si hay carril lateral o rampas |
| `es_lopez_mateos` | 1 si tiene ≥ 30 m de López Mateos |

In [6]:
v = vias.copy()
v["clase"] = v.highway.str.split(";").str[0]
v["es_rampa"] = v.clase.str.endswith("_link")
v["es_lateral"] = v.name.fillna("").str.contains("LATERAL", case=False)
v["base"] = v.name.map(nombre_canonico)
v["carriles"], v["vel_max"] = v.lanes.map(a_numero), v.maxspeed.map(a_numero)
v["puente"] = v.bridge.fillna("").str.contains("yes|viaduct", case=False)
v["tunel"] = v.tunnel.fillna("").str.contains("yes|culvert|building_passage", case=False)

vj = gpd.sjoin(v, cel[["gh7", "geometry"]], predicate="intersects")        # cada tramo con las celdas que toca
vj["largo_m"] = shapely.length(shapely.intersection(vj.geometry.values, cel.set_index("gh7").loc[vj.gh7, "geometry"].values))

troncal = vj[~vj.es_rampa & ~vj.es_lateral & (vj.base != "")]
largo = troncal.groupby(["gh7", "base"]).largo_m.sum().reset_index()
via_principal = largo.sort_values("largo_m").drop_duplicates("gh7", keep="last").set_index("gh7").base.rename("via_principal")
troncal = troncal.join(via_principal, on="gh7")
es_ppal = troncal.base == troncal.via_principal
via_nombre = troncal[es_ppal].groupby("gh7").name.agg(lambda s: s.str.split(";").str[0].mode().iat[0]).rename("via_nombre")

f_via = troncal[es_ppal].groupby("gh7").agg(clase_via=("clase", lambda s: max(ORDEN_CLASE.get(c, 0) for c in s)),
                                            carriles_via=("carriles", "max"), vel_max_via=("vel_max", "max"),
                                            puente=("puente", "any"), tunel=("tunel", "any"))
f_av = vj[~vj.es_rampa].groupby("gh7").agg(n_avenidas=("base", lambda s: s[s != ""].nunique()), lateral=("es_lateral", "any"))
f_av["rampas"] = vj[vj.es_rampa].groupby("gh7").size().reindex(f_av.index).notna()
f_lm = (troncal[troncal.base == "lopez mateos"].groupby("gh7").largo_m.sum() >= 30).rename("es_lopez_mateos")
f_via.head()

,clase_via,carriles_via,vel_max_via,puente,tunel
gh7,,,,,
9ewkmvr,2,NaN,NaN,False,False
9ewkmvv,2,NaN,NaN,False,False
9ewkmvw,2,NaN,NaN,False,False
9ewkmvx,2,NaN,NaN,False,False
9ewkmvy,2,NaN,NaN,False,False


### 3b. Las calles que llegan a la celda

| Variable | Qué es |
|---|---|
| `n_intersecciones` | esquinas donde llegan 3 o más calles |
| `n_semaforos` | semáforos en esas esquinas |
| `n_calles_distintas` | calles con nombre distinto que tocan la celda |
| `cruce_carriles_max` | carriles de la calle más grande que cruza (sin contar la avenida principal) |
| `glorieta` | 1 si hay glorieta |

In [7]:
a = aristas.copy()
a["carriles"], a["base"] = a.lanes.map(a_numero), a.name.map(nombre_canonico)
a["glorieta"] = a.junction.fillna("").str.contains("roundabout|circular", case=False)
aj = gpd.sjoin(a, cel[["gh7", "geometry"]], predicate="intersects").join(via_principal, on="gh7")
f_red = aj.groupby("gh7").agg(n_calles_distintas=("base", lambda s: s[s != ""].nunique()), glorieta=("glorieta", "any"))
f_red["cruce_carriles_max"] = aj[(aj.base != aj.via_principal) & (aj.base != "")].groupby("gh7").carriles.max()

nj = gpd.sjoin(nodos, cel[["gh7", "geometry"]], predicate="within")
f_nod = nj.groupby("gh7").agg(n_intersecciones=("street_count", lambda s: (s >= 3).sum()),
                              n_semaforos=("highway", lambda s: s.fillna("").str.contains("traffic_signals").sum()))
f_nod.head()

,n_intersecciones,n_semaforos
gh7,,
9ewkmyu,4,0
9ewkmzg,4,0
9ewkmzh,1,0
9ewkmzk,3,0
9ewkmzs,3,0


### 3c. El entorno a 100 m
Se agranda cada celda 100 m y se cuentan los puntos de cada tipo: `n_100m_parada`, `n_100m_cruce_peatonal`, `n_100m_semaforo`, `n_100m_comercio`, `n_100m_servicio`.

In [8]:
cel_100 = cel[["gh7", "geometry"]].copy()
cel_100["geometry"] = cel_100.buffer(RADIO_ENTORNO_M)
f_poi = (gpd.sjoin(pois, cel_100, predicate="within").groupby(["gh7", "tipo"]).size()
         .unstack(fill_value=0).add_prefix(f"n_{RADIO_ENTORNO_M}m_"))
f_poi = f_poi[[c for c in f_poi if not c.endswith("_otro")]]
f_poi.head()

tipo,n_100m_comercio,n_100m_cruce_peatonal,n_100m_parada,n_100m_semaforo,n_100m_servicio
gh7,,,,,
9ewkmyu,0,0,1,0,0
9ewkmzg,0,0,4,0,1
9ewkmzh,0,0,4,0,0
9ewkmzk,0,0,5,0,1
9ewkmzs,3,0,6,0,3


### 3d. Se junta todo en una tabla de celdas
Las celdas sin dato en un conteo quedan en 0. Se agregan `semaforo` (1 si hay algún semáforo), `cruce_avenidas` y la distancia al centro.

In [9]:
celdas = cel[["gh7", "lat", "lon"]].set_index("gh7").join([via_principal, via_nombre, f_via, f_av, f_lm, f_red, f_nod, f_poi])
for c in ["puente", "tunel", "lateral", "rampas", "es_lopez_mateos", "glorieta"]:
    celdas[c] = celdas[c].eq(True).astype(int)
conteos = [c for c in celdas if c.startswith("n_")]
celdas[conteos] = celdas[conteos].fillna(0).astype(int)
celdas["semaforo"] = ((celdas.n_semaforos > 0) | (celdas.get("n_100m_semaforo", 0) > 0)).astype(int)
celdas["cruce_avenidas"] = (celdas.n_avenidas >= 2).astype(int)
la, lo, la0, lo0 = np.radians(celdas.lat), np.radians(celdas.lon), np.radians(CENTRO_GDL[0]), np.radians(CENTRO_GDL[1])
celdas["dist_centro_km"] = 6371 * np.arccos(np.clip(np.sin(la) * np.sin(la0) + np.cos(la) * np.cos(la0) * np.cos(lo - lo0), -1, 1))
celdas = celdas.reset_index()
print(f"{len(celdas):,} celdas · {celdas.es_lopez_mateos.sum()} en López Mateos")
celdas.describe().T[["mean", "min", "max"]].round(2)

5,775 celdas · 234 en López Mateos


,mean,min,max
lat,20.64,20.45,20.80
lon,-103.37,-103.52,-103.20
clase_via,1.99,1.00,4.00
carriles_via,2.92,1.00,6.00
vel_max_via,65.20,30.00,90.00
puente,0.03,0.00,1.00
tunel,0.02,0.00,1.00
n_avenidas,1.17,0.00,5.00
lateral,0.08,0.00,1.00
rampas,0.24,0.00,1.00


## Paso 4. Asignar cada accidente a su celda
1. Se busca la **avenida principal más cercana** al accidente. Si está a más de 40 m, el accidente no fue en una avenida principal y se descarta.
2. El accidente se "pega" al punto más cercano de esa avenida, y la celda de ese punto es su celda. Así nunca cae en una celda que no está sobre la avenida.

In [10]:
pts = gpd.GeoDataFrame(acc, geometry=gpd.points_from_xy(acc.LONGITUD, acc.LATITUD), crs="EPSG:4326").to_crs(UTM)
av = avenidas[["geometry"]].assign(avenida=avenidas.name.map(nombre_canonico)).rename_axis("idx_via").reset_index()
j = gpd.sjoin_nearest(pts, av, how="inner", max_distance=BUFFER_VIA_M, distance_col="dist_via_m")
j = j[~j.index.duplicated()]                                    # si empata con dos avenidas, se queda con una
lineas = gpd.GeoSeries(avenidas.geometry.loc[j.idx_via].values, crs=UTM)
pegado = lineas.interpolate(lineas.project(j.geometry.reset_index(drop=True))).to_crs("EPSG:4326")
j["gh7"] = [pgh.encode(y, x, GH_PREC) for x, y in zip(pegado.x, pegado.y)]
j = j[j.gh7.isin(celdas.gh7)]
acc_cel = pd.DataFrame(j.drop(columns=["geometry", "index_right", "idx_via"], errors="ignore"))
acc_cel["grave"] = (acc_cel.CLASE < 3).astype(int)               # con heridos o muertos
print(f"Accidentes en avenidas principales: {len(acc_cel):,} de {len(acc):,} ({len(acc_cel) / len(acc):.0%})")
acc_cel.groupby("gh7").size().sort_values(ascending=False).head(10).rename("accidentes 2022–2024")

Accidentes en avenidas principales: 16,607 de 24,872 (67%)


gh7
9ewmp7y    146
9ewmq2y    101
9ewt2fe     86
9ewmysu     84
9ewmqyw     79
9ewmtzd     75
9ewmx7e     72
9ewmyfj     70
9ewt0u7     68
9ewmq78     67
Name: accidentes 2022–2024, dtype: int64

## Paso 5. Municipio de cada celda y celdas de **buen registro**
- **Municipio de la celda:** el del accidente registrado más cercano a su centro.
- **Buen registro:** algunos municipios registran muchos menos accidentes por habitante (abajo). Lo más probable es que **reporten menos**, no que sean más seguros. Si se entrenara con ellos, el modelo aprendería que sus calles "son seguras". Por eso **solo se modelan las celdas de Guadalajara, Zapopan y Tlaquepaque**.

In [11]:
POBLACION_2020 = {"Guadalajara": 1385629, "Zapopan": 1476491, "Tlaquepaque": 687127, "Tonalá": 569913,
                  "Tlajomulco": 727750, "El Salto": 232852}
tasa = acc.groupby("MUNICIPIO").size().rename("accidentes").to_frame().loc[list(POBLACION_2020)]
tasa["por 100 mil hab. al año"] = (tasa.accidentes / acc.ANIO.nunique() / pd.Series(POBLACION_2020) * 1e5).round(0)
print(tasa.sort_values("por 100 mil hab. al año", ascending=False).to_string(), "\n")

c_pts = gpd.GeoDataFrame(celdas[["gh7"]], geometry=gpd.points_from_xy(celdas.lon, celdas.lat), crs="EPSG:4326").to_crs(UTM)
mp = gpd.sjoin_nearest(c_pts, pts[["MPIO", "MUNICIPIO", "geometry"]], how="left")
mp = mp[~mp.index.duplicated()]
celdas["mpio"], celdas["municipio"] = mp.MPIO.values, mp.MUNICIPIO.values
celdas["buen_registro"] = celdas.mpio.isin(MUNICIPIOS_BUEN_REGISTRO).astype(int)
celdas.groupby("municipio").agg(celdas=("gh7", "size"), buen_registro=("buen_registro", "max")).sort_values("celdas", ascending=False)

             accidentes  por 100 mil hab. al año
MUNICIPIO                                       
Guadalajara       11568                    278.0
Zapopan            8043                    182.0
Tlaquepaque        3067                    149.0
Tonalá              922                     54.0
Tlajomulco         1150                     53.0
El Salto            111                     16.0 



,celdas,buen_registro
municipio,,
Zapopan,1757,1
Guadalajara,1618,1
Tlajomulco,927,0
Tlaquepaque,768,1
Tonalá,389,0
El Salto,310,0
Ixtlahuacán de los Membrillos,6,0


## Paso 6. Clima (Open-Meteo)
Lluvia y temperatura **por hora** de 2022–2024 en el centro de Guadalajara (una sola consulta gratuita). Se resume por día:

| Variable | Qué es |
|---|---|
| `lluvia_mm` | lluvia total del día |
| `lluvia_max_1h` | la hora más lluviosa del día |
| `horas_lluvia` | horas con ≥ 0.5 mm |
| `temp_media` | temperatura promedio |

Es un solo punto para toda la ciudad. Es una limitación: las tormentas en Guadalajara son muy locales.

In [12]:
if not Path("data/clima_horario.csv").exists():
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=120, params={
        "latitude": 20.6774, "longitude": -103.3475, "start_date": "2022-01-01", "end_date": "2024-12-31",
        "hourly": "temperature_2m,precipitation", "timezone": "America/Mexico_City"})
    h = r.json()["hourly"]
    pd.DataFrame({"fecha_hora": h["time"], "temperatura": h["temperature_2m"],
                  "precipitacion_mm": h["precipitation"]}).to_csv("data/clima_horario.csv", index=False)
clima_h = pd.read_csv("data/clima_horario.csv", parse_dates=["fecha_hora"])
clima_d = clima_h.groupby(clima_h.fecha_hora.dt.normalize().rename("FECHA")).agg(
    lluvia_mm=("precipitacion_mm", "sum"), lluvia_max_1h=("precipitacion_mm", "max"),
    horas_lluvia=("precipitacion_mm", lambda x: (x >= .5).sum()), temp_media=("temperatura", "mean")).round(2)
clima_d.to_csv("data/clima_diario.csv")
print(f"{len(clima_h):,} horas → {len(clima_d):,} días")
clima_d.describe().round(1)

26,304 horas → 1,096 días


,lluvia_mm,lluvia_max_1h,horas_lluvia,temp_media
count,1096.0,1096.0,1096.0,1096.0
mean,1.6,0.8,0.7,20.8
std,4.0,1.9,1.8,2.9
min,0.0,0.0,0.0,13.3
25%,0.0,0.0,0.0,19.2
50%,0.0,0.0,0.0,20.7
75%,0.6,0.4,0.0,22.4
max,34.6,14.3,20.0,28.6


## Guardar

In [13]:
celdas.to_csv("data/celdas.csv", index=False)
acc_cel.to_csv("data/accidentes_celdas.csv", index=False)
print(f"→ data/celdas.csv            {len(celdas):,} celdas ({celdas.buen_registro.sum():,} de buen registro)")
print(f"→ data/accidentes_celdas.csv {len(acc_cel):,} accidentes con su celda")
print(f"→ data/clima_diario.csv      {len(clima_d):,} días")

→ data/celdas.csv            5,775 celdas (4,143 de buen registro)
→ data/accidentes_celdas.csv 16,607 accidentes con su celda
→ data/clima_diario.csv      1,096 días
